In [1]:
# =============================================================================
# VALIDATION DE L'ENVIRONNEMENT DE SIMULATION (PortfolioEnvironment)
# =============================================================================
# === CELLULE 1: IMPORTS ET CONFIGURATION ===
import os
import sys
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

print("--- Initialisation du banc d'essai pour PortfolioEnvironment ---")

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

try:
    from src.data_manager import BRVMTrainingManager
    from src.environment import PortfolioEnvironment
    print("✅ Importations réussies.")
except ImportError as e:
    print(f"❌ ERREUR D'IMPORTATION: {e}")
    raise

# === CELLULE 2: CHARGEMENT DES DONNÉES ===
print("\n--- Chargement des données pré-traitées ---")
try:
    base_path = '..'
    processed_data_path = os.path.join(base_path, 'processed_data')

    with open(os.path.join(processed_data_path, 'all_data.pkl'), 'rb') as f:
        all_data = pickle.load(f)
    with open(os.path.join(processed_data_path, 'dividendes.pkl'), 'rb') as f:
        dividendes = pickle.load(f)
    with open(os.path.join(processed_data_path, 'fundamentals_panel.pkl'), 'rb') as f:
        fundamentals_panel = pickle.load(f)
    print("✅ Données chargées avec succès.")
except Exception as e:
    print(f"❌ ERREUR: {e}")
    raise

# === CELLULE 3: PRÉPARATION DES DONNÉES POUR L'ENVIRONNEMENT ===
print("\n--- Préparation des données pour une simulation de test ---")

manager = BRVMTrainingManager(
    all_data=all_data,
    dividendes=dividendes,
    fundamentals_panel=fundamentals_panel,
    rebalancing_freq_days=7  # Fréquence hebdomadaire pour un test rapide
)

# Génération des données pour la période 'test' (plus courte)
test_topk_dates = manager.generate_topk_dates_for_period('test', K=10)
test_indicators = manager.generate_indicators_for_period('test', test_topk_dates)

print(f"✅ Données de test prêtes: {len(test_topk_dates)} dates de rééquilibrage.")

# === CELLULE 4: INITIALISATION DE L'ENVIRONNEMENT ===
print("\n--- Création et validation de l'environnement ---")

try:
    env = PortfolioEnvironment(
        all_data=all_data,
        topk_dates=test_topk_dates,
        indicators_topk=test_indicators,
        fundamentals_panel=fundamentals_panel,
        initial_cash=10_000_000,
        transaction_cost=0.01,
        alpha_cvar=0.95,
        k_assets=10,
        use_copula=False,  # Désactivé pour ce test
        copula_function=None
    )
    print("✅ Environnement créé avec succès!")
    print(f"   - Espace d'observation: {env.observation_space}")
    print(f"   - Espace d'action: {env.action_space}")
    print(f"   - Nombre d'actifs: {env.k_assets}")
except Exception as e:
    print(f"❌ ERREUR lors de la création de l'environnement: {e}")
    raise

# === CELLULE 5: TEST DU CYCLE DE VIE (reset et step) ===
print("\n" + "="*60)
print("🧪 TEST DU CYCLE DE VIE DE L'ENVIRONNEMENT")
print("="*60)

# Test 1: reset()
print("\n--- Test du reset() ---")
obs, info = env.reset()
print(f"   ✅ reset() réussi.")
print(f"   - Dimension de l'observation: {obs.shape}")
print(f"   - Valeur totale initiale: {info['total_value']:,.0f} FCFA")
print(f"   - Date initiale: {info['current_date'].date()}")

# Test 2: Exécution de 5 steps avec des actions aléatoires
print("\n--- Test de 5 steps avec des actions aléatoires ---")
for i in range(5):
    print(f"\n--- Step {i+1} ---")

    # Génération d'une action aléatoire valide (somme à 1)
    random_action = np.random.dirichlet(np.ones(env.k_assets))
    print(f"   - Action aléatoire: {np.round(random_action, 3)}")

    obs, reward, terminated, truncated, info = env.step(random_action)
    done = terminated or truncated

    print(f"   - Récompense: {reward:.4f}")
    print(f"   - Valeur totale: {info['total_value']:,.0f} FCFA")
    print(f"   - Épisode terminé: {done}")

    if done:
        print("   - Épisode terminé prématurément.")
        break

# === CELLULE 6: VISUALISATION DES TRANSACTIONS ===
print("\n" + "="*60)
print("📊 ANALYSE DES TRANSACTIONS")
print("="*60)

# Réinitialisation pour analyser les transactions
obs, info = env.reset()
initial_value = info['total_value']

# Exécution d'un step avec une action spécifique
test_action = np.array([0.1, 0.2, 0.3, 0.1, 0.1, 0.1, 0.05, 0.03, 0.01, 0.01])
test_action = test_action / np.sum(test_action)  # Normalisation

print(f"\n--- Test avec une action spécifique: {np.round(test_action, 3)} ---")
obs, reward, terminated, truncated, info = env.step(test_action)

print(f"\n📊 RÉSULTATS:")
print(f"   - Récompense: {reward:.4f}")
print(f"   - Valeur initiale: {initial_value:,.0f} FCFA")
print(f"   - Valeur finale: {info['total_value']:,.0f} FCFA")
print(f"   - Rendement: {(info['total_value']/initial_value - 1)*100:+.2f}%")
print(f"   - Cash: {info['cash']:,.0f} FCFA")
print(f"   - Poids des actifs: {info['weights']}")

# === CELLULE 7: TEST DES PÉNALITÉS DE TRANSACTION ===
print("\n" + "="*60)
print("💰 TEST DES COÛTS DE TRANSACTION")
print("="*60)

# Réinitialisation
obs, info = env.reset()
initial_cash = info['cash']

# Action qui déclenche des transactions (changement complet des poids)
new_action = np.zeros(env.k_assets)
new_action[0] = 1.0  # Tout dans le premier actif

print(f"\n--- Action déclenchant des transactions: {new_action} ---")
obs, reward, terminated, truncated, info = env.step(new_action)

print(f"\n📊 COÛTS DE TRANSACTION:")
print(f"   - Cash initial: {initial_cash:,.0f} FCFA")
print(f"   - Cash final: {info['cash']:,.0f} FCFA")
print(f"   - Coût total: {initial_cash - info['cash']:,.0f} FCFA")
print(f"   - Coût en % du portefeuille: {(initial_cash - info['cash'])/initial_value*100:+.2f}%")

# === CELLULE 8: FERMETURE ===
env.close()
print("\n✅ TEST DE L'ENVIRONNEMENT TERMINÉ AVEC SUCCÈS!")


--- Initialisation du banc d'essai pour PortfolioEnvironment ---
✅ Importations réussies.

--- Chargement des données pré-traitées ---
✅ Données chargées avec succès.

--- Préparation des données pour une simulation de test ---
🏛️ BRVMTrainingManager initialisé (fréquence: 7 jours).

📅 Génération des dates de rééquilibrage pour 'TEST'...
   -> 53 dates générées.
Pré-traitement des données pour le Top-K...


Préparation des prix:   0%|          | 0/45 [00:00<?, ?it/s]

Génération de la liste Top-K:   0%|          | 0/53 [00:00<?, ?it/s]

✅ Top-K généré : 53 dates avec 10 actifs sélectionnés


Calcul des indicateurs:   0%|          | 0/53 [00:00<?, ?it/s]

✅ Indicateurs générés : 530 observations pour la période 'test'
✅ Données de test prêtes: 53 dates de rééquilibrage.

--- Création et validation de l'environnement ---
✅ Environnement créé avec succès!
   - Espace d'observation: Box(-inf, inf, (497,), float32)
   - Espace d'action: Box(0.0, 1.0, (10,), float32)
   - Nombre d'actifs: 10

🧪 TEST DU CYCLE DE VIE DE L'ENVIRONNEMENT

--- Test du reset() ---
   ✅ reset() réussi.
   - Dimension de l'observation: (497,)
   - Valeur totale initiale: 10,000,000 FCFA
   - Date initiale: 2024-01-02

--- Test de 5 steps avec des actions aléatoires ---

--- Step 1 ---
   - Action aléatoire: [0.27  0.019 0.069 0.168 0.03  0.001 0.066 0.062 0.226 0.088]

==================== DÉBUT STEP 1 (2024-01-02) ====================
[ENV] Actifs sélectionnés: ['ONTBF', 'BOAB', 'SIBC', 'ECOC', 'BOAC', 'BOAN', 'SNTS', 'ETIT', 'BOABF', 'ORAC']
[ENV] Valeur initiale: 10,000,000 FCFA
[ENV] Phase 1: Vente des actifs non sélectionnés...
[ENV] NAV disponible: 10,000,000 